### 1. Read files & Basic setting

In [1]:
import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns

# Output
output_dir = r"C:\Users\user\Downloads"
os.makedirs(output_dir, exist_ok=True)

# Display
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option('display.float_format', '{:.2f}'.format)

# CSV
users_df = pd.read_csv(r'C:\Users\user\OneDrive\Desktop\Chrissy\code\archive\users_data.csv')
transactions_df = pd.read_csv(r'C:\Users\user\OneDrive\Desktop\Chrissy\code\archive\transactions_data.csv')
cards_df = pd.read_csv(r'C:\Users\user\OneDrive\Desktop\Chrissy\code\archive\cards_data.csv')

import json

# json_train_fraud_labels
with open(r'C:\Users\user\OneDrive\Desktop\Chrissy\code\archive\train_fraud_labels.json') as f:
    train_fraud_labels = json.load(f)
if 'target' in train_fraud_labels:
    train_fraud_labels_dict = train_fraud_labels['target']
else:
    train_fraud_labels_dict = train_fraud_labels

train_fraud_labels_df = pd.DataFrame(
    list(train_fraud_labels_dict.items()),
    columns=['transaction_id', 'fraud_label']
)

# json_mcc
with open(r'C:\Users\user\OneDrive\Desktop\Chrissy\code\archive\mcc_codes.json') as f:
    mcc_codes = json.load(f)
mcc_codes_df = pd.DataFrame.from_dict(mcc_codes, orient='index').reset_index()
mcc_codes_df.columns = ['mcc_code', 'category_name']

### 2. Data Preprocessing

#### 2.1 rename columns

In [2]:
# transactions
transactions_df = transactions_df.rename(columns={'mcc': 'mcc_code'})
transactions_df = transactions_df.rename(columns={'id': 'transaction_id'})

# users
users_df = users_df.rename(columns={'id':'client_id'})

# cards
cards_df = cards_df.rename(columns={'id':'card_id'})

#### 2.2 missing value

In [3]:
def check_missing(df, df_name, auto_add_flag=True):
    missing_info = df.isnull().sum()
    missing_info = missing_info[missing_info > 0]

    if missing_info.empty:
        print(f"{df_name}: 無缺值")
    else:
        missing_percentage = (missing_info / len(df)) * 100
        missing_df = pd.DataFrame({
            'Missing Count': missing_info,
            'Missing Percentage': missing_percentage
        }).sort_values(by='Missing Count', ascending=False)

        print(f"{df_name}: 有缺值")
        print(missing_df)
        print("-" * 40)

        if auto_add_flag:
            for col in missing_info.index:
                flag_col = f"{col}_missing_flag"
                if flag_col not in df.columns:
                    df[flag_col] = df[col].isna().astype(int)
                    print(f"已新增缺失指標欄位: {flag_col}")


check_missing(transactions_df, "transactions_df")
check_missing(cards_df, "cards_df")
check_missing(train_fraud_labels_df, "train_fraud_labels_df")
check_missing(users_df, "users_df")
check_missing(mcc_codes_df, "mcc_codes_df")

transactions_df: 有缺值
                Missing Count  Missing Percentage
errors               13094522               98.41
zip                   1652706               12.42
merchant_state        1563700               11.75
----------------------------------------
已新增缺失指標欄位: merchant_state_missing_flag
已新增缺失指標欄位: zip_missing_flag
已新增缺失指標欄位: errors_missing_flag
cards_df: 無缺值
train_fraud_labels_df: 無缺值
users_df: 無缺值
mcc_codes_df: 無缺值


#### 2.3 missing value imputation

In [4]:
# merchant_state
transactions_df['merchant_city'] = transactions_df['merchant_city'].astype('category')
transactions_df.loc[
    transactions_df['merchant_city'].str.lower() == 'online', 'merchant_state'
] = 'online'

# zip (20250927)
transactions_df.loc[
    transactions_df['zip'].isna() & (transactions_df['merchant_city'].str.lower() == 'online'),
    'zip'
] = -1

transactions_df.loc[
    transactions_df['zip'].isna() & (transactions_df['merchant_city'].str.lower() != 'online'),
    'zip'
] = -999

# errors
transactions_df['errors'] = transactions_df['errors'].astype('category')
transactions_df['errors'] = transactions_df['errors'].cat.add_categories('No error').fillna('No error')

#### 2.4 format conversion

In [5]:
import pandas as pd
import json

# transactions
transactions_df['date'] = pd.to_datetime(transactions_df['date'])
transactions_df['amount'] = transactions_df['amount'].replace(r'[\$,]', '', regex=True).astype(float).astype(int)
transactions_df['use_chip'] = transactions_df['use_chip'].astype('category')
transactions_df['merchant_state'] = transactions_df['merchant_state'].astype('category')
transactions_df['mcc_code'] = transactions_df['mcc_code'].astype('int64')
transactions_df['zip'] = transactions_df['zip'].astype('int64') #20250927

# user
users_df['per_capita_income'] = [int(i[1:]) for i in users_df['per_capita_income']]
users_df['yearly_income'] = [int(i[1:]) for i in users_df['yearly_income']]
users_df['total_debt'] = [int(i[1:]) for i in users_df['total_debt']]
users_df['gender'] = users_df['gender'].astype('category')

# cards
cards_df['expires'] = pd.to_datetime(cards_df['expires'], format='%m/%Y')
cards_df['acct_open_date'] = pd.to_datetime(cards_df['acct_open_date'], format='%m/%Y')
cards_df['credit_limit']= [int(i[1:]) for i in cards_df['credit_limit']]
cards_df['card_brand'] = cards_df['card_brand'].astype('category')
cards_df['card_type'] = cards_df['card_type'].astype('category')
cards_df['has_chip'] = cards_df['has_chip'].map({'YES': 1, 'NO': 0})
cards_df['has_chip'] = cards_df['has_chip'].astype('int64')

# train_fraud_labels
train_fraud_labels_df['transaction_id'] = train_fraud_labels_df['transaction_id'].astype('int64')
train_fraud_labels_df['fraud_label'] = train_fraud_labels_df['fraud_label'].map({'Yes': True, 'No': False})

# MCC
mcc_codes_df['mcc_code'] = mcc_codes_df['mcc_code'].astype('int64')

#### 2.5 one hot encoding

In [6]:
# cards_df
cols_to_encode = ['card_type', 'card_brand']
dummies_cards = pd.get_dummies(cards_df[cols_to_encode], prefix=cols_to_encode, dtype='uint8')
cards_df = pd.concat([cards_df, dummies_cards], axis=1)

# transactions_df
dummies_chip = pd.get_dummies(transactions_df['use_chip'], prefix='use_chip', dtype='uint8')
transactions_df = pd.concat([transactions_df, dummies_chip], axis=1)

### 3. dataset integration

In [7]:
# dataset integration #

# fraud
int_transactions_df = transactions_df.merge(train_fraud_labels_df, on='transaction_id', how='inner')
#int_transactions_df = transactions_df.merge(train_fraud_labels_df, on='transaction_id', how='left')


# users
int_transactions_df = int_transactions_df.merge(users_df, on='client_id', how='left')

# cards
int_transactions_df = int_transactions_df.merge(
    cards_df,
    left_on=['card_id', 'client_id'],
    right_on=['card_id', 'client_id'],
    how='left'
)

check_missing(int_transactions_df, "int_transactions_df")

# MCC(Optional)
int_transactions_df = int_transactions_df.merge(mcc_codes_df, on='mcc_code', how='left')

del users_df
del transactions_df
del cards_df
del train_fraud_labels_df
del mcc_codes_df

int_transactions_df: 無缺值


### 4. Feature engineering

#### 4.1 recency

In [8]:
# RecencyInterval
int_transactions_df = int_transactions_df.sort_values(by=['client_id', 'date'])
int_transactions_df['RecencyInterval'] = int_transactions_df.groupby('client_id')['date'].diff().dt.total_seconds()/60
int_transactions_df['RecencyInterval'] = int_transactions_df['RecencyInterval'].fillna(0)

#### 4.2 frequency

In [9]:
import numpy as np

int_transactions_df = int_transactions_df.sort_values(by=['client_id', 'date']).reset_index(drop=True)

windows = [7, 30, 60, 90]

for w in windows:
    int_transactions_df[f'TxnFrequency_{w}d'] = 0

for client, group in int_transactions_df.groupby('client_id', sort=False):
    dates = group['date'].values
    n = len(dates)
    
    for w in windows:
        left = 0
        freq = np.zeros(n, dtype=int)
        delta = np.timedelta64(w, 'D')
        
        for right in range(n):
            while dates[right] - dates[left] > delta:
                left += 1
            freq[right] = right - left
        
        int_transactions_df.loc[group.index, f'TxnFrequency_{w}d'] = freq



#### 4.3 monetary

In [10]:
import numpy as np
import pandas as pd

int_transactions_df = int_transactions_df.sort_values(by=['client_id', 'date']).reset_index(drop=True)

int_transactions_df['AmtDelta'] = np.nan

for client, group in int_transactions_df.groupby('client_id', sort=False):
    amounts = group['amount'].values
    n = len(amounts)
    mean_prev = np.zeros(n)
    
    cumsum = np.cumsum(amounts)
    mean_prev[1:] = cumsum[:-1] / np.arange(1, n)
    
    amt_delta = amounts - mean_prev
    
    int_transactions_df.loc[group.index, 'AmtDelta'] = amt_delta


#### 4.4 dk_location

In [11]:
# grouping

us_region_map = {
    'Northeast': ['NY','NJ','PA','MA','CT','RI','NH','VT','ME'],
    'Midwest': ['IL','OH','MI','IN','WI','MN','IA','MO','ND','SD','NE','KS'],
    'South': ['FL','GA','SC','NC','AL','MS','LA','TX','OK','TN','KY','VA','WV','AR','MD','DE','DC'],
    'West': ['CA','WA','OR','NV','AZ','NM','CO','UT','ID','MT','WY','AK','HI'],
}

continent_map = {
    'North America': [
        'United States','Canada','Mexico','The Bahamas','Dominican Republic','Jamaica',
        'Costa Rica','Haiti','Belize','Barbados','Trinidad and Tobago','Aruba',
        'Antigua and Barbuda','Guatemala','Honduras','Panama','Saint Vincent and the Grenadines'
    ],
    'Europe': [
        'United Kingdom','France','Germany','Italy','Spain','Netherlands','Switzerland','Ireland',
        'Sweden','Norway','Denmark','Finland','Austria','Belgium','Portugal','Poland','Greece',
        'Czech Republic','Hungary','Romania','Slovenia','Slovakia','Serbia','Bosnia and Herzegovina',
        'Croatia','Lithuania','Latvia','Estonia','Iceland','Malta','Monaco','Luxembourg','Andorra',
        'Vatican City','Ukraine','Russia','Turkey',
        'Albania','Cyprus','Kosovo','Macedonia','Moldova','Montenegro'
    ],
    'Asia': [
        'China','Japan','South Korea','India','Taiwan','Hong Kong','Thailand','Singapore','Malaysia',
        'Philippines','Indonesia','Vietnam','Pakistan','Bangladesh','Sri Lanka','Myanmar (Burma)',
        'Nepal','Saudi Arabia','United Arab Emirates','Israel','Qatar','Bahrain','Iran','Jordan',
        'Lebanon','Kuwait','Georgia','Mongolia','Kazakhstan',
        'Azerbaijan','Brunei','East Timor (Timor-Leste)','Iraq','Kyrgyzstan','Maldives','Oman','Uzbekistan','Yemen'
    ],
    'South America': [
        'Brazil','Peru','Chile','Argentina','Colombia','Uruguay','Ecuador','Venezuela','Guyana','Suriname'
    ],
    'Africa': [
        'South Africa','Nigeria','Ghana','Kenya','Morocco','Egypt','Ethiopia','Senegal','Liberia','Mozambique',
        'Zambia','Zimbabwe',"Cote d'Ivoire",'Cameroon','Sudan','Tunisia','Algeria','Mali','Benin','Burkina Faso',
        'Eritrea','Guinea','Niger','Togo',
        'Cabo Verde','Equatorial Guinea','Gabon','Republic of the Congo','Seychelles','Sierra Leone','South Sudan','Swaziland'
    ],
    'Oceania': [
        'Australia','New Zealand','Fiji','Nauru','Papua New Guinea','Vanuatu','Samoa','Micronesia','Solomon Islands',
        'Marshall Islands','Tuvalu',
        'Tonga'
    ],
    'Online': ['online','AA']
}


us_region_lookup = {state: region for region, states in us_region_map.items() for state in states}
continent_lookup = {country: cont for cont, countries in continent_map.items() for country in countries}


int_transactions_df['merchant_online'] = int_transactions_df['merchant_state'].eq('online')
int_transactions_df['merchant_us'] = int_transactions_df['merchant_state'].isin(us_region_lookup.keys())
int_transactions_df['merchant_eu'] = int_transactions_df['merchant_state'].isin(continent_map['Europe'])
int_transactions_df['merchant_others'] = ~( 
    int_transactions_df['merchant_online'] |
    int_transactions_df['merchant_us'] |
    int_transactions_df['merchant_eu']
)

#### 4.5 dk_others

In [12]:
int_transactions_df = int_transactions_df.sort_values(['client_id', 'date']).reset_index(drop=True)
int_transactions_df['FirstTxnInRegion'] = ~int_transactions_df.duplicated(subset=['client_id', 'merchant_state'])
int_transactions_df['FirstTxnInRegion'] = int_transactions_df['FirstTxnInRegion'].astype('uint8')

In [13]:
df = int_transactions_df
TARGET_COL = 'fraud_label'

def calculate_fraud_rate_by_category(df, column_name):
    temp_df = df[[column_name, TARGET_COL]].dropna()
    fraud_rate_per_category = (
        temp_df.groupby(column_name)[TARGET_COL]
        .mean() * 100
    ).reset_index()
    fraud_rate_per_category.columns = [column_name, 'fraud_rate']
    return fraud_rate_per_category

fraud_rate_df = calculate_fraud_rate_by_category(int_transactions_df, 'mcc_code')

fraud_rate_df.sort_values('fraud_rate', ascending=False)

high_risk_mcc = fraud_rate_df[fraud_rate_df['fraud_rate'] > 20]['mcc_code'].tolist()


In [14]:
df = int_transactions_df
TARGET_COL = 'fraud_label'

def calculate_fraud_rate_by_category(df, column_name):
    temp_df = df[[column_name, TARGET_COL]].dropna()

    fraud_rate_per_category = (
        temp_df.groupby(column_name)[TARGET_COL]
        .mean() * 100
    ).reset_index()
    
    fraud_rate_per_category.columns = [column_name, 'fraud_rate']
    return fraud_rate_per_category

fraud_rate_df = calculate_fraud_rate_by_category(int_transactions_df, 'mcc_code')

#fraud_rate_df = fraud_rate_df.sort_values('fraud_rate', ascending=False)

high_risk_MCC_list = fraud_rate_df.loc[fraud_rate_df['fraud_rate'] > 2, 'mcc_code'].tolist()

int_transactions_df['HighRiskMCC'] = int_transactions_df['mcc_code'].isin(high_risk_MCC_list)
int_transactions_df['HighRiskMCC'] = int_transactions_df['HighRiskMCC'].astype('uint8')

In [15]:
# TxnToLimitRatio
#int_transactions_df['TxnToLimitRatio'] = int_transactions_df['amount'] / int_transactions_df['credit_limit']
#int_transactions_df['TxnToLimitRatio'] = int_transactions_df['TxnToLimitRatio'].replace([np.inf, -np.inf], 0)

### 5. preparation

#### 5.1 result df

In [16]:
import pandas as pd

result = pd.DataFrame(columns=[
    "Model", "Features", 
    "Train AUC", "Test AUC", 
    "Train PR AUC", "Test PR AUC"
])

#### 5.2 grouping for features

In [17]:
# Raw features
raw_cols_n = [
    'amount', 'current_age', 'retirement_age',
    'latitude', 'longitude', 'per_capita_income', 'yearly_income',
    'total_debt', 'credit_score', 'num_credit_cards', 'has_chip',
    'num_cards_issued', 'credit_limit', 'year_pin_last_changed'
]

raw_cols_c =[
    'use_chip','merchant_city','merchant_state',
    'gender','card_brand', 'card_type',
]

# RFM features
rfm_cols = [
    'RecencyInterval', 'TxnFrequency_7d','TxnFrequency_30d',
    'TxnFrequency_60d', 'TxnFrequency_90d','AmtDelta'
]

# DK features
dk_cols = [
    'merchant_online', 'merchant_us', 'merchant_eu', 'merchant_others',
    'FirstTxnInRegion', 'HighRiskMCC'
]
#'FirstTxnInRegion',

# Others
# one-hot
#'use_chip_Chip Transaction', 'use_chip_Online Transaction', 'use_chip_Swipe Transaction',
#'card_type_Credit', 'card_type_Debit', 'card_type_Debit (Prepaid)',
#'card_brand_Amex', 'card_brand_Discover', 'card_brand_Mastercard', 'card_brand_Visa',
#'error_category_Authentication Error', 'error_category_Card Info Error', 'error_category_Multiple Errors', 'error_category_No Error', 'error_category_Payment Failure',

# Grouping
feature_groups_lr = {
    "X_raw": raw_cols_n,
    "X_rfm": rfm_cols,
    "X_dk": dk_cols,
    "X_raw + X_rfm": raw_cols_n + rfm_cols,
    "X_raw + X_dk": raw_cols_n + dk_cols,
    "X_rfm + X_dk": rfm_cols + dk_cols,
    "X_raw + X_rfm + X_dk": raw_cols_n + rfm_cols + dk_cols
}

feature_groups_xgb = {
    "X_raw": raw_cols_n + raw_cols_c,
    "X_rfm": rfm_cols,
    "X_dk": dk_cols,
    "X_raw + X_rfm": raw_cols_n + raw_cols_c + rfm_cols,
    "X_raw + X_dk": raw_cols_n + raw_cols_c + dk_cols,
    "X_rfm + X_dk": rfm_cols + dk_cols,
    "X_raw + X_rfm + X_dk": raw_cols_n + raw_cols_c + rfm_cols + dk_cols
}

### 6. model function

#### 6.1 lr function 

In [30]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

def run_lr(df, feature_list, label_col, feature_name):
    global result

    X = df[feature_list]
    y = df[label_col]

    # train/test split
    df = df.sort_values('date').reset_index(drop=True)
    split_idx = int(len(df) * 0.8)

    X_train = df.loc[:split_idx-1, feature_list]
    X_test  = df.loc[split_idx:, feature_list]
    y_train = df.loc[:split_idx-1, label_col]
    y_test  = df.loc[split_idx:, label_col]

    # scaling
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    # model
    lr = LogisticRegression(max_iter=2000)
    lr.fit(X_train_s, y_train)

    # predict prob
    train_pred = lr.predict_proba(X_train_s)[:, 1]
    test_pred  = lr.predict_proba(X_test_s)[:, 1]

    # metrics
    train_auc = roc_auc_score(y_train, train_pred)
    test_auc  = roc_auc_score(y_test, test_pred)

    train_pr  = average_precision_score(y_train, train_pred)
    test_pr   = average_precision_score(y_test, test_pred)

    # append
    result.loc[len(result)] = [
        "Logistic Regression", 
        feature_name,
        train_auc, test_auc,
        train_pr, test_pr
    ]


#### 6.2 xgb function

In [31]:
from xgboost import XGBClassifier
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.model_selection import train_test_split

# XGBoost function
def run_xgb(df, feature_list, label_col, feature_name):
    global result

    X = df[feature_list]
    y = df[label_col]

    # train/test split
    df = df.sort_values('date').reset_index(drop=True)
    split_idx = int(len(df) * 0.8)

    X_train = df.loc[:split_idx-1, feature_list]
    X_test  = df.loc[split_idx:, feature_list]
    y_train = df.loc[:split_idx-1, label_col]
    y_test  = df.loc[split_idx:, label_col]

    # model
    xgb_model = XGBClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="logloss",
        tree_method="hist",
        enable_categorical=True
    )
    xgb_model.fit(X_train, y_train)

    # predict probabilities
    train_pred = xgb_model.predict_proba(X_train)[:,1]
    test_pred  = xgb_model.predict_proba(X_test)[:,1]

    # metrics
    train_auc = roc_auc_score(y_train, train_pred)
    test_auc  = roc_auc_score(y_test, test_pred)

    train_pr  = average_precision_score(y_train, train_pred)
    test_pr   = average_precision_score(y_test, test_pred)

    # append
    result.loc[len(result)] = [
        "XGBoost",
        feature_name,
        train_auc, test_auc,
        train_pr, test_pr
    ]

#### 6.3 lgbm function

In [32]:
def run_lgbm(df, feature_list, label_col, feature_name):
    global result

    X = df[feature_list]
    y = df[label_col]

    # train/test split
    df = df.sort_values('date').reset_index(drop=True)
    split_idx = int(len(df) * 0.8)

    X_train = df.loc[:split_idx-1, feature_list]
    X_test  = df.loc[split_idx:, feature_list]
    y_train = df.loc[:split_idx-1, label_col]
    y_test  = df.loc[split_idx:, label_col]

    # model
    lgb_model = lgb.LGBMClassifier(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        class_weight='balanced'
    )
    lgb_model.fit(X_train, y_train)

    # predict probabilities
    train_pred = lgb_model.predict_proba(X_train)[:,1]
    test_pred  = lgb_model.predict_proba(X_test)[:,1]

    # metrics
    train_auc = roc_auc_score(y_train, train_pred)
    test_auc  = roc_auc_score(y_test, test_pred)

    train_pr  = average_precision_score(y_train, train_pred)
    test_pr   = average_precision_score(y_test, test_pred)

    # append
    result.loc[len(result)] = [
        "LightGBM",
        feature_name,
        train_auc, test_auc,
        train_pr, test_pr
    ]

In [33]:
for group_name, group_cols in feature_groups_lr.items():
    run_lr(int_transactions_df, group_cols, "fraud_label", group_name)

for group_name, group_cols in feature_groups_xgb.items():
    run_xgb(int_transactions_df, group_cols, "fraud_label", group_name)
    run_lgbm(int_transactions_df, group_cols, "fraud_label", group_name)


[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Info] Number of positive: 10215, number of negative: 7121755
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.778382 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5665
[LightGBM] [Info] Number of data points in the train set: 7131970, number of used features: 20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with po

In [34]:
result

,Model,Features,Train AUC,Test AUC,Train PR AUC,Test PR AUC
0,Logistic Regression,X_raw,0.67,0.67,0.01,0.01
1,Logistic Regression,X_rfm,0.74,0.74,0.01,0.01
2,Logistic Regression,X_dk,0.92,0.92,0.10,0.11
3,Logistic Regression,X_raw + X_rfm,0.74,0.75,0.01,0.01
4,Logistic Regression,X_raw + X_dk,0.93,0.93,0.14,0.16
5,Logistic Regression,X_rfm + X_dk,0.95,0.95,0.16,0.18
6,Logistic Regression,X_raw + X_rfm + X_dk,0.95,0.95,0.17,0.19
7,XGBoost,X_raw,0.97,0.97,0.40,0.41
8,LightGBM,X_raw,0.95,0.94,0.34,0.33
9,XGBoost,X_rfm,0.82,0.81,0.02,0.02


## 🔬 兩次模型實驗結果差異分析與問題診斷 from gemini

本分析基於您提供的兩次實驗數據（第一次：索引 0-20；第二次：索引 21-41）以及您所做的三項關鍵變動。

### 🎯 實驗條件變動對結果的影響分析

#### 變動 1：以時間順序切分資料 (80% 訓練, 20% 測試)

| 指標 | 第一次實驗 (無時間切分) | 第二次實驗 (時間切分) | 診斷與推論 |
| :--- | :--- | :--- | :--- |
| **Test AUC** | 普遍高且穩定 ($0.90$ 以上) | **普遍更高** ($0.97$ - $0.99$) | 整體資料（多數類）的結構在時間上可能相對穩定，模型區分能力提升。 |
| **Test PR AUC** | 樹模型最高可達 **0.48** (ID 19) | 樹模型最高僅達 **0.35** (ID 36) | **強烈證據顯示正類別（少數類）的行為模式隨時間發生了漂移 (Data Drift)**，導致模型在新近的測試資料上對少數類的預測能力大幅下降。 |

#### 變動 3：新增 LightGBM 中 `class_weight="balanced"` 參數

| 模型 | 實驗次數 | `class_weight` 狀態 | Test PR AUC 表現 | 診斷結果 |
| :--- | :--- | :--- | :--- | :--- |
| LightGBM | 第一次 (ID 20) | 未啟用 | 0.46 | 表現優秀。 |
| LightGBM | 第二次 (ID 37, 41) | 已啟用 | 0.20 / 0.18 | 表現大幅退步。 |
| **總結** | N/A | N/A | **`class_weight="balanced"` 的正面效果被時間切分帶來的數據漂移所抵消甚至掩蓋。** 模型即使被強制關注少數類，但學到的舊模式在新資料上仍是無效的。 |

---

### 📈 總結診斷：PR-AUC 下降的主因

第二次實驗的 Test AUC 雖然優異（最高達 0.99），但 Test PR AUC 大幅下降的主要原因歸咎於：

> **核心問題：正類別隨時間的特徵漂移 (Class-Specific Feature Drift)**
> 訓練集和測試集之間的時間差異，使得模型在訓練集中學到的少數類別模式，與測試集中的新模式不符。

---

### 🛠️ 實際解決方案與改善建議

為了在時間序列切分下最大化您的 PR-AUC 效能，建議採取以下步驟：

#### 1. 強化對抗時間漂移的能力（優先）

* **檢查正類別的分佈漂移：**
    * 在訓練集（前 80%）和測試集（後 20%）中，單獨觀察正類別樣本的關鍵特徵（如 `X_raw`, `X_rfm`, `X_dk` 的平均值或中位數）是否發生顯著變化。
* **納入時間相關特徵：**
    * 在特徵工程中加入能捕捉時間趨勢的變量（例如：最近 $N$ 天的平均值、與上週相比的增長率等）。
* **滑動時間窗訓練：**
    * 考慮採用**滑動時間窗**的方式進行驗證，讓訓練資料更接近測試資料的時間點，確保模型能夠學習到最新的少數類模式。

#### 2. 針對 PR-AUC 指標進行客製化最佳化

由於簡單的 `class_weight="balanced"` 無效，應轉向**手動調校**：

* **手動調校類別權重 (Manual Class Weight Tuning)：**
    * 不要使用 `balanced`，而是手動設定少數類權重 $W$，並將 $W$ 作為超參數進行網格搜尋。
    * 公式：$W = \frac{\text{總樣本數}}{\text{正類別數}}$（作為起始點），然後在其周圍進行更精細的搜尋。
* **閾值最佳化 (Threshold Optimization)：**
    * 模型的預測分數輸出是連續的機率值。傳統上以 $0.5$ 作為分類閾值。
    * 在訓練完成後，計算不同機率閾值（例如從 $0.05$ 到 $0.5$）下的 Test PR AUC，並選擇能使 PR AUC 達到最佳的**閾值**來進行最終分類。

#### 3. 專注於第二次的最佳基線模型

* 以第二次實驗中表現最好的 **XGBoost, X\_raw + X\_dk (ID 36)** (Test AUC 0.99, Test PR AUC 0.35) 作為基線。
* 對此模型進行上述的 **類別權重調整** 和 **閾值最佳化**，目標是將 Test PR AUC 推升至 $0.48$ 以上。